# Solutions – Day 25 Exercises

In [ ]:
import torch
import faiss
import numpy as np
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from pathlib import Path
import requests
from io import BytesIO

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## Exercise 1: Expand dataset and rebuild index

In [ ]:
# Download 20 more images from a dataset or local folder
# For demonstration, we'll reuse the existing 5 and assume you add more
image_dir = Path("expanded_images")
image_dir.mkdir(exist_ok=True)

# Example: download a few more
extra_urls = [
    "https://images.pexels.com/photos/20787/pexels-photo.jpg",  # mountain
    "https://images.pexels.com/photos/1640777/pexels-photo-1640777.jpeg",  # pizza
]
for i, url in enumerate(extra_urls):
    img = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    img.save(image_dir / f"extra_{i}.jpg")

# Collect all image paths
all_paths = list(Path("search_images").glob("*.jpg")) + list(image_dir.glob("*.jpg"))

# Generate embeddings
embeddings = []
for path in all_paths:
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs).cpu().numpy()
        embeddings.append(emb)
embeddings = np.vstack(embeddings).astype('float32')
faiss.normalize_L2(embeddings)

# Build index
new_index = faiss.IndexFlatIP(embeddings.shape[1])
new_index.add(embeddings)
print(f"Index built with {new_index.ntotal} images")

## Exercise 2: Image‑to‑image accuracy

In [ ]:
def search_image(query_path, index, all_paths, k=3):
    img = Image.open(query_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        q_emb = model.get_image_features(**inputs).cpu().numpy().astype('float32')
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k)
    return [(all_paths[i], scores[0][j]) for j, i in enumerate(indices[0])]

results = search_image("search_images/cat.jpg", new_index, all_paths, k=3)
print("Query: cat.jpg")
for path, score in results:
    print(f"{path.name}: {score:.4f}")
# Expect other animal or textured images

## Exercise 3: Precision at k

In [ ]:
query_text = "dog"
inputs = processor(text=[query_text], return_tensors="pt", padding=True).to(device)
with torch.no_grad():
    t_emb = model.get_text_features(**inputs).cpu().numpy().astype('float32')
faiss.normalize_L2(t_emb)
scores, indices = new_index.search(t_emb, 5)

# Manually label relevance: assume dog.jpg is relevant, maybe also some other
relevant = {"dog.jpg"}
retrieved = [all_paths[i].name for i in indices[0]]
precision_at_3 = len([r for r in retrieved[:3] if r in relevant]) / 3
precision_at_5 = len([r for r in retrieved[:5] if r in relevant]) / 5
print(f"Precision@3: {precision_at_3}")
print(f"Precision@5: {precision_at_5}")

## Exercise 4: Save and load index

In [ ]:
faiss.write_index(new_index, "image_index.faiss")
import pickle
with open("image_paths.pkl", "wb") as f:
    pickle.dump(all_paths, f)

# Later load
loaded_index = faiss.read_index("image_index.faiss")
with open("image_paths.pkl", "rb") as f:
    loaded_paths = pickle.load(f)
print(f"Loaded index with {loaded_index.ntotal} images")

## Exercise 5: Gradio UI (conceptual code)

In [ ]:
# Install: pip install gradio
import gradio as gr

def search_fn(query, image=None):
    if image is not None:
        # image query
        results = search_image(image, loaded_index, loaded_paths, k=3)
    else:
        # text query
        # (implement text search similarly)
        pass
    # return images and scores

iface = gr.Interface(
    fn=search_fn,
    inputs=[gr.Textbox(label="Text query"), gr.Image(type="filepath", label="Or upload image")],
    outputs=gr.Gallery(label="Results"),
    title="Multimodal Search Engine"
)
# iface.launch()